# Week 16 · Notebook 1 — LoRA / PEFT Fine-Tuning Demo
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Fine-tune a **tiny open model** with **LoRA** (low-rank adapters) on a few instruction examples, using the Hugging Face stack — **no OpenAI**.

> **Run on a GPU:** Runtime → Change runtime type → **T4 GPU**.
> Every heavy import is guarded with try/except, so the notebook degrades gracefully (clear message) if a GPU or library is missing.

## 0. Install the Hugging Face fine-tuning stack

In [ ]:
!pip -q install transformers datasets peft accelerate bitsandbytes 2>/dev/null
print('installed (or already present)')

## 1. Check the environment
We confirm a GPU is available and report VRAM. The demo still *builds* without one, but real training needs a GPU.

In [ ]:
HAS_GPU = False
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    if HAS_GPU:
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'GPU: {name}  ~{vram:.1f} GB VRAM')
    else:
        print('No GPU detected. Switch the Colab runtime to a T4 GPU to train.')
except Exception as e:
    print('PyTorch not available yet:', e)

## 2. Memory math (why we use LoRA + quantization)
A rough rule for *loading* a model for inference:
- FP16 ~2 GB / billion params · 8-bit ~1 GB/B · 4-bit ~0.5 GB/B.
Training needs far more (gradients + optimizer + activations) — so we freeze the base model and train only small **LoRA adapters**.

In [ ]:
def loading_gb(params_billions, bits=16):
    return params_billions * (bits / 8) * 2  # ~2 bytes/param at 16-bit

for b in [16, 8, 4]:
    print(f'7B model @ {b}-bit: ~{loading_gb(7, b):.1f} GB to load')

## 3. A tiny instruction dataset
Quality > quantity. Real projects use hundreds of clean, consistent examples; here we use a handful so the demo runs fast.

In [ ]:
train_examples = [
    {'instruction': 'Summarize in one sentence.',
     'input': 'The mitochondria is the powerhouse of the cell.',
     'output': 'Mitochondria produce most of the cell\'s energy.'},
    {'instruction': 'Summarize in one sentence.',
     'input': 'Python is a high-level, readable, general-purpose language.',
     'output': 'Python is a readable general-purpose programming language.'},
    {'instruction': 'Summarize in one sentence.',
     'input': 'A GPU runs many matrix operations in parallel.',
     'output': 'GPUs do parallel matrix math, ideal for neural networks.'},
]
def format_example(ex):
    return (f"### Instruction: {ex['instruction']}\n"
            f"### Input: {ex['input']}\n"
            f"### Response: {ex['output']}")
print(format_example(train_examples[0]))

## 4. Load a small base model + attach a LoRA adapter
We use a tiny open model so it fits on a free Colab GPU. The base weights are **frozen**; only the LoRA `A`/`B` matrices train (often <1% of params).

In [ ]:
MODEL_ID = 'sshleifer/tiny-gpt2'  # tiny open model — fast demo
model = tokenizer = None
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
    lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                      target_modules=['c_attn'], task_type='CAUSAL_LM')
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()  # see the <1% figure
except Exception as e:
    print('Could not load model/PEFT (no GPU or libs?):', e)
    print('That is OK — the rest of the notebook still explains the workflow.')

## 5. Tokenize and train for a couple of epochs
We keep `num_train_epochs` tiny so it finishes quickly. Watch the **loss** drop. If validation loss rises while training loss falls, you are **overfitting**.

In [ ]:
try:
    from datasets import Dataset
    from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
    texts = [format_example(e) for e in train_examples]
    ds = Dataset.from_dict({'text': texts})
    def tok(batch):
        return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=64)
    ds = ds.map(tok, batched=True, remove_columns=['text'])
    collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
    args = TrainingArguments(output_dir='out', num_train_epochs=3,
        per_device_train_batch_size=1, learning_rate=2e-4,
        logging_steps=1, report_to=[])
    trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)
    trainer.train()
    print('Training complete.')
except Exception as e:
    print('Skipping training (no GPU/model):', e)

## 6. Save just the adapter (megabytes, not gigabytes)
You ship the small adapter and load it on top of the same base model. You can keep many task adapters for one base model.

In [ ]:
try:
    model.save_pretrained('my_lora_adapter')
    import os
    files = os.listdir('my_lora_adapter')
    print('Adapter saved:', files)
except Exception as e:
    print('Nothing to save (training did not run):', e)

## 7. Takeaways
- LoRA trains a tiny fraction of params and saves a small **adapter**.
- **QLoRA** loads the frozen base in 4-bit so even big models fit one GPU.
- Fine-tune only after prompting and RAG fall short.
- Always keep a validation split and watch for **overfitting**.